# 02 — Silver Transformation (thin wrapper)

Cleans, types and deduplicates Bronze into `workspace.silver.*` via the pure
functions in `src/transforms.py`, then runs the DQ-01..DQ-07 gate from
`src/dq_checks.py` on the resulting tables.

* renames: Phase→phase, Sponsor→sponsor, Country→country, City→city, Sex→sex
* casts: dates → DATE, numerics → INT
* subjects: dedup by subject_id; blank site_id → `subjects_quarantine`
* visits: orphan subject_id → `visits_quarantine` (null actual_date is valid)

In [0]:
# --- Bootstrap: make the repo's src/ package importable + config ---
import sys

from databricks.sdk.runtime import dbutils

try:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().value()
    _root = "/" + "/".join(_nb.strip("/").split("/")[:-2])
except Exception:
    _root = "/Users/sreekanthreddy2000@gmail.com/clinical-trial-dq-pip"
sys.path.insert(0, "/Workspace" + _root)

import datetime

from src.transforms import (
    transform_studies, transform_sites, transform_subjects, transform_visits,
)

BRONZE, SILVER = "workspace.bronze", "workspace.silver"
SILVER_TS = datetime.datetime.now(datetime.UTC).isoformat()
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")
print(f"Silver transformation started at {SILVER_TS}")

In [0]:
# --- Build Silver via the src functions + MERGE upserts ---
def upsert(df, table, key, delete_not_matched_by_source=False):
    df.createOrReplaceTempView("v_src")
    spark.sql(f"CREATE TABLE IF NOT EXISTS {SILVER}.{table} USING DELTA "
             f"AS SELECT * FROM v_src WHERE 1=0")
    extra = " WHEN NOT MATCHED BY SOURCE THEN DELETE" if delete_not_matched_by_source else ""
    spark.sql(f"""MERGE INTO {SILVER}.{table} t USING v_src s ON t.{key} = s.{key}
                  WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *{extra}""")
    print(f"{table}: {spark.table(f'{SILVER}.{table}').count()} rows (upsert)")

upsert(transform_studies(spark.table(f"{BRONZE}.studies"), SILVER_TS), "studies", "study_id")
upsert(transform_sites(spark.table(f"{BRONZE}.sites"), SILVER_TS), "sites", "site_id")

subjects_valid, subjects_quar = transform_subjects(spark.table(f"{BRONZE}.subjects"), SILVER_TS)
upsert(subjects_valid, "subjects", "subject_id")
upsert(subjects_quar, "subjects_quarantine", "subject_id")

visits_valid, visits_quar = transform_visits(
    spark.table(f"{BRONZE}.visits"), spark.table(f"{SILVER}.subjects"), SILVER_TS)
upsert(visits_valid, "visits", "visit_id", delete_not_matched_by_source=True)
upsert(visits_quar, "visits_quarantine", "visit_id")

In [0]:
# --- DQ gate: DQ-01..DQ-07 on the Silver tables (src/dq_checks.py) ---
from src.dq_checks import all_passed, run_dq_checks, summarize

checks = run_dq_checks(
    subjects=spark.table(f"{SILVER}.subjects"),
    visits=spark.table(f"{SILVER}.visits"),
    studies=spark.table(f"{SILVER}.studies"),
    sites=spark.table(f"{SILVER}.sites"),
)
print(summarize(checks))
assert all_passed(checks), "Silver DQ gate failed — see summary above"

In [0]:
# --- Bronze -> Silver reconciliation ---
print("=" * 60)
print("=== BRONZE -> SILVER RECONCILIATION ===")
print("=" * 60)
print(f"{'Table':<10}{'Bronze':>8}{'Silver':>8}{'Quarantine':>12}")
for t in ["studies", "sites", "subjects", "visits"]:
    b = spark.table(f"{BRONZE}.{t}").count()
    s = spark.table(f"{SILVER}.{t}").count()
    q = spark.table(f"{SILVER}.{t}_quarantine").count() if t in ("subjects", "visits") else 0
    print(f"{t:<10}{b:>8}{s:>8}{q:>12}")